# polars-umap: UMAP inside Polars

[polars-umap](https://polars-umap.jeroenjanssens.com) computes UMAP embeddings *inside* Polars expressions. This notebook shows what changes compared to the usual workflow with [umap-learn](https://umap-learn.readthedocs.io), using the same MNIST data as `01-mnist.ipynb`: every row is a 28 × 28 grayscale image of a handwritten digit, flattened to 784 pixel columns.

It needs a few extra packages:

```sh
uv sync --group polars-umap   # polars-umap, umap-learn and matplotlib
```

In [ ]:
import gzip
import urllib.request
from pathlib import Path

import numpy as np

CACHE = Path.home() / ".cache" / "positron-atlas" / "datasets"


def download(url: str, name: str) -> Path:
    """Download a file once and cache it."""
    path = CACHE / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {url} ...")
        tmp = path.with_suffix(path.suffix + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    return path


def read_idx(path: Path) -> np.ndarray:
    """Read an IDX file (the MNIST format): images as (n, 784) uint8, labels as (n,)."""
    with gzip.open(path, "rb") as f:
        data = f.read()
    ndim = data[3]
    shape = tuple(int.from_bytes(data[4 + 4 * i : 8 + 4 * i], "big") for i in range(ndim))
    arr = np.frombuffer(data, dtype=np.uint8, offset=4 + 4 * ndim)
    return arr.reshape(shape[0], -1) if ndim > 1 else arr

In [ ]:
import time

import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs

LABELS = [str(i) for i in range(10)]
BASE = "https://ossci-datasets.s3.amazonaws.com/mnist/"
files = {
    name: download(BASE + name, "mnist-" + name)
    for name in [
        "train-images-idx3-ubyte.gz",
        "train-labels-idx1-ubyte.gz",
        "t10k-images-idx3-ubyte.gz",
        "t10k-labels-idx1-ubyte.gz",
    ]
}
pixels = np.concatenate(
    [read_idx(files["train-images-idx3-ubyte.gz"]), read_idx(files["t10k-images-idx3-ubyte.gz"])]
)
labels = np.concatenate(
    [read_idx(files["train-labels-idx1-ubyte.gz"]), read_idx(files["t10k-labels-idx1-ubyte.gz"])]
)

mnist = pl.from_numpy(
    (pixels / 255).astype(np.float32), schema=[f"px{i}" for i in range(pixels.shape[1])]
).with_columns(
    pl.Series("label", [LABELS[v] for v in labels], dtype=pl.Enum(LABELS)),
    pl.Series("split", np.where(np.arange(len(labels)) < 60_000, "train", "test")),
)
mnist.shape

MNIST has ten digits, more than color can tell apart reliably. The plots below color the digits 3, 5 and 8, which are easily confused, and show the others in gray.

In [ ]:
DIGIT_COLORS = {"3": "#2a78d6", "5": "#eb6834", "8": "#1baf7a"}


def plot(df: pl.DataFrame, x: str = "umap_0", y: str = "umap_1", size: float = 2, title: str = ""):
    """Scatter plot of an embedding, with the digits 3, 5 and 8 colored."""
    fig, ax = plt.subplots(figsize=(6, 5))
    other = df.filter(~pl.col("label").is_in(list(DIGIT_COLORS)))
    ax.scatter(other[x], other[y], s=size, c="#d4d3cf", linewidths=0, label="other")
    for digit, color in DIGIT_COLORS.items():
        part = df.filter(pl.col("label") == digit)
        ax.scatter(part[x], part[y], s=size, c=color, linewidths=0, label=digit)
    ax.set_axis_off()
    ax.set_title(title, loc="left")
    ax.legend(markerscale=4 * (2 / size) ** 0.5, frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()


# A random sample of 10,000 digits, to keep umap-learn quick.
digits = mnist.sample(10_000, seed=1)

## Before: convert, compute, bind

With umap-learn, a UMAP embedding of a DataFrame takes three steps. You convert the feature columns to a NumPy array, fit the model, and bind the coordinates back to the DataFrame:

In [ ]:
import warnings

import umap

# Silence umap-learn's warnings about its default pipeline, and about running on
# one thread because of random_state.
warnings.filterwarnings("ignore", module="umap")

t0 = time.perf_counter()

# 1. Convert the pixel columns to a NumPy array
X = digits.select(cs.starts_with("px")).to_numpy()

# 2. Compute the embedding
embedding = umap.UMAP(random_state=42).fit_transform(X)

# 3. Bind the coordinates back to the data (and hope the rows still line up)
before = digits.with_columns(
    pl.Series("umap_0", embedding[:, 0]),
    pl.Series("umap_1", embedding[:, 1]),
)

print(f"{time.perf_counter() - t0:.1f} s")
plot(before, title="umap-learn")

The first call includes numba's just-in-time compilation, which adds several seconds in every new Python session. Run the cell again to see the time of the fit itself. Also, setting `random_state` makes umap-learn run on a single thread.

Every change to the data means repeating all three steps. Filtering `digits` without filtering `embedding` misaligns the rows, and a separate embedding per group needs a loop.

## After: one expression

With polars-umap, the embedding is an expression. Importing `polars_umap` adds a `umap` namespace to DataFrames, LazyFrames and expressions. `fit_transform()` takes column names, expressions or selectors, here all pixel columns, and adds the components as `umap_0` and `umap_1`:

In [ ]:
import polars_umap as pu

t0 = time.perf_counter()
after = digits.umap.fit_transform(cs.starts_with("px"), random_state=42)
print(f"{time.perf_counter() - t0:.1f} s")

plot(after, title="polars-umap")

There's no conversion and no binding: the coordinates are computed for exactly the rows of `digits`, in the same order. With `random_state`, the result is the same on every run, regardless of the number of threads.

The same as an expression, which you can combine with anything else in `select()` or `with_columns()`. It returns one `Array(Float32, 2)` column:

In [ ]:
digits.select("label", xy=pu.umap(cs.starts_with("px"), random_state=42)).head()

## All 70,000 digits

polars-umap is fast enough for the whole data set:

In [ ]:
t0 = time.perf_counter()
mnist = mnist.umap.fit_transform(cs.starts_with("px"), random_state=42)
print(f"{time.perf_counter() - t0:.1f} s")

plot(mnist, size=0.5, title="All 70,000 MNIST digits")

`mnist` now has `umap_0` and `umap_1` columns, so you can open it in the Data Explorer and color by any column.

## Things that are easy now

Because the embedding is an expression, the rest of Polars works with it.

**One embedding per group.** `.over()` fits a separate UMAP for every group, in a single query. Here the training and test images each get their own embedding:

In [ ]:
per_split = digits.with_columns(
    xy=pu.umap(cs.starts_with("px"), random_state=42).over("split")
).with_columns(umap_0=pl.col("xy").arr.get(0), umap_1=pl.col("xy").arr.get(1))

for split in ["train", "test"]:
    plot(per_split.filter(pl.col("split") == split), title=f"{split} split, embedded on its own")

**Unusual digits.** `knn()` exposes the nearest neighbor search that UMAP uses. The mean distance to the 10 nearest neighbors in pixel space is large for digits that don't look like any other. (Every row is its own first neighbor, so ask for 11 and skip the first.)

In [ ]:
unusual = (
    digits.with_columns(nn=pu.knn(cs.starts_with("px"), k=11, random_state=42))
    .with_columns(knn_dist=pl.col("nn").struct.field("distances").arr.to_list().list.slice(1).list.mean())
    .sort("knn_dist", descending=True)
)

fig, axes = plt.subplots(1, 8, figsize=(8, 1.4))
for ax, row in zip(axes, unusual.head(8).iter_rows(named=True)):
    ax.imshow(np.array([row[f"px{i}"] for i in range(784)]).reshape(28, 28), cmap="gray_r")
    ax.set_title(f"label {row['label']}", fontsize=9)
    ax.set_axis_off()
plt.show()

**Lazy queries.** The `umap` namespace works on LazyFrames too, so filtering, embedding and aggregating are one query. Here only the digits 3, 5 and 8 are embedded:

In [ ]:
(
    mnist.lazy()
    .filter(pl.col("label").is_in(["3", "5", "8"]))
    .umap.fit_transform(cs.starts_with("px"), random_state=42, prefix="sub_")
    .group_by("label")
    .agg(pl.col("sub_0", "sub_1").median())
    .sort("label")
    .collect()
)

## Benchmark

polars-umap is 2–12× faster than umap-learn, and the Polars layer adds no measurable overhead over the Rust implementation from Apple's [embedding-atlas](https://github.com/apple/embedding-atlas) that it uses. All implementations find embeddings of the same quality:

![Fit time of polars-umap, polars-umap on the GPU, the upstream Rust binary and umap-learn, for six datasets](assets/polars-umap-benchmark.png)

MNIST (70,000 × 784), umap-learn 0.6 pipeline, median of 3 runs on an Apple M4 Pro:

| Implementation | Fit time | vs. umap-learn | Trustworthiness | Label kNN accuracy |
|---|---:|---:|---:|---:|
| polars-umap | 2.4 s | 3.4× | 0.950 | 0.970 |
| polars-umap (GPU) | 2.2 s | 3.7× | 0.950 | 0.968 |
| upstream Rust binary | 2.5 s | 3.4× | 0.950 | 0.970 |
| umap-learn | 8.3 s | 1.0× | 0.951 | 0.968 |

These times exclude umap-learn's numba compilation, which adds another 10–12 s to the first call. Trustworthiness measures how well the embedding preserves the nearest neighbors of the input. See the [benchmarks page](https://polars-umap.jeroenjanssens.com/user-guide/benchmarks.html) for all datasets, both pipelines, memory use and the methodology.